# Intelligent Asset Lifecycle Management
## Random Forest Failure Prediction

Final Random Forest pipeline for predicting `failure_within_24h`.

**Pipeline:** Load → Clean → Feature Engineering → Split → Preprocess → Random Forest → Evaluate → Cross-Validate → Overfitting Analysis → Feature Importance → Save Model


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")
print("Libraries imported successfully.")


## 1. Load Dataset

In [ ]:
DATA_PATH = "industrial_machine_predictive_maintenance.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()


## 2. Inspect Dataset

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
print(df["failure_within_24h"].value_counts())


## 3. Clean and Feature Engineer

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

if "machine_id" in df.columns:
    df = df.drop(columns=["machine_id"])

if {"temperature_motor", "ambient_temp"}.issubset(df.columns):
    df["temperature_rise"] = (
        df["temperature_motor"] - df["ambient_temp"]
    )

if {"current_phase_avg", "rpm"}.issubset(df.columns):
    df["electrical_load_index"] = (
        df["current_phase_avg"] * df["rpm"]
    )

print("Shape after cleaning:", df.shape)
df.head()


## 4. Features and Target

Target: `failure_within_24h`

Excluded to avoid target leakage:

- `rul_hours`
- `failure_type`
- `estimated_repair_cost`


In [ ]:
TARGET = "failure_within_24h"

leakage_columns = [
    TARGET,
    "rul_hours",
    "failure_type",
    "estimated_repair_cost"
]

X = df.drop(
    columns=[c for c in leakage_columns if c in df.columns]
)
y = df[TARGET].astype(int)

print("Features:")
print(X.columns.tolist())
print("\nX shape:", X.shape)


## 5. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


## 6. Preprocessing

In [ ]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)


## 7. Random Forest

We avoid a huge GridSearchCV. This configuration is intentionally chosen to reduce the extreme complexity seen with unlimited tree depth while keeping the model strong.


In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", rf)
])

model.fit(X_train, y_train)

print("Training completed.")


## 8. Training vs Testing Accuracy

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("Training Accuracy:", round(train_accuracy, 4))
print("Testing Accuracy :", round(test_accuracy, 4))
print("Accuracy Gap     :", round(train_accuracy - test_accuracy, 4))


## 9. Full Test Evaluation

In [ ]:
y_test_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, y_test_pred), 4))
print("Precision:", round(precision_score(y_test, y_test_pred, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, y_test_pred, zero_division=0), 4))
print("F1 Score :", round(f1_score(y_test, y_test_pred, zero_division=0), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_test_prob), 4))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["No Failure", "Failure"],
    zero_division=0
))


## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

print(cm)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["No Failure", "Failure"])
plt.yticks([0, 1], ["No Failure", "Failure"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.show()


## 11. 5-Fold Cross-Validation

This checks how consistently the model performs across different training subsets.

The held-out test set is not used here.


In [ ]:
cv_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

print("CV scores:", cv_scores)
print("Mean CV accuracy:", round(cv_scores.mean(), 4))
print("CV standard deviation:", round(cv_scores.std(), 4))


## 12. Generalization Summary

In [ ]:
print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Mean CV Accuracy  : {cv_scores.mean():.4f}")
print(f"Test Accuracy     : {test_accuracy:.4f}")
print(f"CV Std            : {cv_scores.std():.4f}")
print(f"Training-CV Gap   : {train_accuracy - cv_scores.mean():.4f}")


## 13. Training vs Testing Accuracy by Tree Depth

In [ ]:
depths = [2, 4, 6, 8, 10, 15, 20, 25, None]
depth_train_scores = []
depth_test_scores = []

for depth in depths:
    rf_depth = RandomForestClassifier(
        n_estimators=200,
        max_depth=depth,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    depth_model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", rf_depth)
    ])

    depth_model.fit(X_train, y_train)

    depth_train_scores.append(
        accuracy_score(y_train, depth_model.predict(X_train))
    )
    depth_test_scores.append(
        accuracy_score(y_test, depth_model.predict(X_test))
    )

plt.figure(figsize=(9, 5))
plt.plot(range(len(depths)), depth_train_scores, marker="o", label="Training Accuracy")
plt.plot(range(len(depths)), depth_test_scores, marker="o", label="Testing Accuracy")
plt.xticks(range(len(depths)), [str(d) for d in depths])
plt.xlabel("Maximum Tree Depth")
plt.ylabel("Accuracy")
plt.title("Random Forest: Training vs Testing Accuracy")
plt.legend()
plt.grid(True)
plt.show()


## 14. Feature Importance

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
importances = model.named_steps["model"].feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

feature_importance_df.head(20)


In [ ]:
top_features = feature_importance_df.head(15)

plt.figure(figsize=(9, 6))
plt.barh(
    top_features["Feature"][::-1],
    top_features["Importance"][::-1]
)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top Random Forest Features")
plt.show()


## 15. Generate Failure Risk

In [ ]:
asset_predictions = X_test.copy()

asset_predictions["actual_failure"] = y_test.values
asset_predictions["failure_probability"] = y_test_prob
asset_predictions["predicted_failure"] = y_test_pred

def risk_level(probability):
    if probability >= 0.75:
        return "CRITICAL"
    elif probability >= 0.50:
        return "HIGH"
    elif probability >= 0.25:
        return "MODERATE"
    return "LOW"

asset_predictions["risk_level"] = [
    risk_level(p) for p in y_test_prob
]

asset_predictions.sort_values(
    "failure_probability",
    ascending=False
).head(20)


## 16. Save Final Model

In [ ]:
MODEL_PATH = "random_forest_asset_failure_model.joblib"

joblib.dump(model, MODEL_PATH)

print("Saved:", MODEL_PATH)


# Final Output

The saved pipeline predicts:

- Failure / no failure
- Failure probability
- Asset risk level

This becomes **Model 1** of the lifecycle system. It can later be combined with Model 2 (RUL prediction), maintenance history, repair cost, and asset criticality for the final lifecycle decision engine.
